In [2]:
from types import resolve_bases
# import matplotlib.pyplot as plt
import pickle
import numpy as np
import pandas as pd
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
import xgboost as xgb
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from types import resolve_bases
from sklearn.metrics import mean_squared_log_error
from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from hyperopt.early_stop import no_progress_loss
import copy

# Test/Train Split

This section of code obtains a test/train split that does not suffer from test/train leakage by spatial proximity.

In [3]:
GrdSrch_df = pd.read_csv("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/Project-1_BO4BT/ExperimentalSeries-1_PGXOpt/Part-C_HypOptComp_PGCI/Part-A1_PGCI-GrdSrch-[27]-P3O1/raw-data_2023-03-10_PtA1-PGCI-GrdSrch-[27]-P3O1_Stykke-4.csv")
RndSrch_df = pd.read_csv("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/Project-1_BO4BT/ExperimentalSeries-1_PGXOpt/Part-C_HypOptComp_PGCI/Part-A2_PGCI-RndSrch-[27]-P3O1/raw-data_2023-03-10_PtA2-PGCI-RndSrch-[27]-P3O1_Stykke-4.csv")
GrdSrch_df.rename(columns={"x1": "s1", "x2": "s2", "x3": "b1", "delta_polymer_mass_pct": "DeltaPolymerMass_pct", "polymer_start_mass_g": "StartPolymerMass_g", "polymer_end_mass_pct": "EndPolymerMass_pct"},inplace=True)
RndSrch_df.rename(columns={"x1": "s1", "x2": "s2", "x3": "b1", "delta_polymer_mass_pct": "DeltaPolymerMass_pct", "polymer_start_mass_g": "StartPolymerMass_g", "polymer_end_mass_pct": "EndPolymerMass_pct"},inplace=True)
BOpt_8SP_3It_df = pd.read_csv("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/Project-1_BO4BT/ExperimentalSeries-1_PGXOpt/Part-C_HypOptComp_PGCI/Part-B5_PGCI-BOpt-[8,3,3,3,3,3,3,1]-P3O1/raw-data_2023-03-20_PtB5-PGCI-BOpt-[8,3,3,3,3,3,3,1]-P3O1-Stykke-4.csv")
BOpt_8SP_3It_df.rename(columns={"x1": "s1", "x2": "s2", "x3": "b1", "delta_polymer_mass_pct": "DeltaPolymerMass_pct", "polymer_start_mass_g": "StartPolymerMass_g", "polymer_end_mass_pct": "EndPolymerMass_pct"},inplace=True)

In [4]:
df = pd.concat(objs=[GrdSrch_df,RndSrch_df,BOpt_8SP_3It_df])
df.drop(columns=["mould_position","G_stoichiometry","CA_stoichiometry","IA_stoichiometry","StartPolymerMass_g","EndPolymerMass_pct"],inplace=True)
df['DeltaPolymerMass_pct']=df['DeltaPolymerMass_pct']*-1
df

,s1,s2,b1,DeltaPolymerMass_pct
0,0.8000,1.0000,0.8000,13.360997
1,1.0000,0.4000,0.6000,12.188133
2,0.4000,0.6000,0.8000,14.405027
3,0.8000,0.8000,1.0000,10.823033
4,0.4000,0.8000,0.6000,14.041169
...,...,...,...,...
24,0.4413,0.9805,0.5755,11.473868
25,0.4925,0.8819,0.5811,12.517877
26,0.3829,0.6579,0.7756,16.666833
27,0.3349,0.6460,0.7712,17.279822


In [5]:
def PredictorsToCaStoichs(s1,b1):
    return (0.5+(2.0-0.5)*s1)*b1

def PredictorsToIaStoichs(s2,b1):
    return (1.0+(2.0-1.0)*s2)*(1-b1)

In [6]:
CaStoichs = []
for s1,b1 in zip(df["s1"],df["b1"]):
    CaStoichs.append(PredictorsToCaStoichs(s1,b1))
IaStoichs = []
for s2,b1 in zip(df["s2"],df["b1"]):
    IaStoichs.append(PredictorsToIaStoichs(s2,b1))
CaStoichs = np.array(CaStoichs)
IaStoichs = np.array(IaStoichs)

In [7]:
df = df.drop(columns=["s1","s2","b1"])
df["n_ci"] = CaStoichs
df["n_it"] = IaStoichs
df

,DeltaPolymerMass_pct,n_ci,n_it
0,13.360997,1.360000,0.400000
1,12.188133,1.200000,0.560000
2,14.405027,0.880000,0.320000
3,10.823033,1.700000,0.000000
4,14.041169,0.660000,0.720000
...,...,...,...
24,11.473868,0.668702,0.840722
25,12.517877,0.719838,0.788328
26,16.666833,0.833266,0.372033
27,17.279822,0.773012,0.376605


In [8]:
# Plotting the Data
plt.scatter(df["n_ci"],df["n_it"],c=df["DeltaPolymerMass_pct"])

NameError: name 'plt' is not defined

In [9]:
X = df[["n_ci","n_it"]].to_numpy()
y = df["DeltaPolymerMass_pct"].to_numpy()

In [10]:
# Displaying the finally selected seed.
# 3863 is pretty good, fulfills all criteria well.
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.33,random_state=3863)
plt.scatter(X_train.T[0],X_train.T[1],c=y_train)
plt.scatter(X_test.T[0],X_test.T[1],c="red",s=1)

NameError: name 'plt' is not defined

# Naive Training

In [11]:
tree_methods = ['approx', 'exact', 'hist']

space = {
'tree_method': hp.choice('tree_method', tree_methods),
'max_depth': hp.randint('max_depth', 3, 12),
'n_estimators': hp.randint('n_estimators', 1, 1000),
'num_parallel_tree': hp.randint('num_parallel_tree', 2, 8),
'min_child_weight': hp.randint('min_child_weight', 1, 200),
'subsample': hp.uniform('subsample', 0.7, 1),
'colsample_bytree': hp.uniform('colsample_bytree', 0.5, 1),
'colsample_bylevel': hp.uniform('colsample_bylevel', 0.5, 1),
'reg_lambda': hp.uniform('reg_lambda', 0, 10),
'learning_rate': hp.uniform('learning_rate', 0, 1),
}

def objective(params):
    xgb_model = xgb.XGBRegressor(**params)
    xgb_model.fit(X_train, y_train)
    y_pred = xgb_model.predict(X_test)
    MSLE = mean_squared_log_error(y_test,y_pred)
    return {'loss': MSLE, 'status': STATUS_OK}

trials = Trials()

best_params = fmin(objective, space, algo=tpe.suggest, max_evals=50, trials=trials)
best_params["tree_method"] = tree_methods[best_params["tree_method"]]
print("Best set of hyperparameters: ", best_params)

100%|██████████| 50/50 [00:18<00:00,  2.64trial/s, best loss: 0.014493263559314863]
Best set of hyperparameters:  {'colsample_bylevel': np.float64(0.5382627351621087), 'colsample_bytree': np.float64(0.5720863350996936), 'learning_rate': np.float64(0.11467356684093077), 'max_depth': np.int64(5), 'min_child_weight': np.int64(8), 'n_estimators': np.int64(918), 'num_parallel_tree': np.int64(2), 'reg_lambda': np.float64(0.6739091929671326), 'subsample': np.float64(0.7092528791554434), 'tree_method': 'hist'}


In [12]:
xgb_model = xgb.XGBRegressor(**best_params)
xgb_model.fit(X_train, y_train)
y_pred = xgb_model.predict(X_test)

rmse = root_mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
print(f"RMSE =\t{round(rmse,3)}")
print(f"MAE =\t{round(mae,3)}")

RMSE =	1.503
MAE =	1.117


In [13]:
plt.scatter(y_test,y_pred)
plt.plot([0,17],[0,17])
plt.ylim(0,17)
plt.xlim(0,17)
plt.xlabel(r"Observations, $y_{test}$")
plt.ylabel(r"Predictions, $y_{pred}$")

NameError: name 'plt' is not defined

In [14]:
y_residuals = y_test-y_pred
plt.scatter(y_test,y_residuals)
# plt.plot([0,17],[0,17])
plt.xlabel(r"Observations, $y_{test}$")
plt.ylabel(r"Residuals, $y_{test}-y_{pred}$")

NameError: name 'plt' is not defined

In [15]:
n = 25
iCoords_arr = np.linspace(0,2,n-1)
jCoords_arr = np.linspace(0,2,n-1)

ijCoords_lis = []
for i in iCoords_arr:
    for j in jCoords_arr:
        ijCoords_lis.append([i,j])
ijCoords_arr = np.array(ijCoords_lis)
y_pred_arr = xgb_model.predict(ijCoords_arr)

n_ci_bits = np.array(ijCoords_lis).T[0]
n_it_bits = np.array(ijCoords_lis).T[1]
plt.scatter(x=n_ci_bits,y=n_it_bits,c=y_pred_arr,cmap='viridis')

NameError: name 'plt' is not defined

# k-fold Cross-Evaluation

In [16]:
xgb_model = xgb.XGBRegressor(**best_params)
cv = KFold(n_splits=4,shuffle=True,random_state=1)
kfold_neg_mae = cross_val_score(estimator=xgb_model,X=X_train,y=y_train,scoring='neg_mean_absolute_error',cv=cv,n_jobs=-1)
kfold_mae = kfold_neg_mae * -1
print(f"MAE =\t{round(np.average(kfold_mae),2)}")

MAE =	1.12


In [17]:
xgb_model = xgb.XGBRegressor(**best_params)
xgb_model.fit(X_train, y_train)
y_pred = xgb_model.predict(X_test)

rmse = root_mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
print(f"RMSE =\t{round(rmse,3)}")
print(f"MAE =\t{round(mae,3)}")

RMSE =	1.503
MAE =	1.117


# k-fold Cross-Evaluation During Training

In [18]:
# X_train, X_test, y_train, y_test
X_trainer = copy.deepcopy(X_train)
y_trainer = copy.deepcopy(y_train)
X_tester = copy.deepcopy(X_test)
y_tester = copy.deepcopy(y_test)

In [19]:
tree_methods = ['approx', 'exact', 'hist']

space = {
'tree_method': hp.choice('tree_method', tree_methods),
'max_depth': hp.randint('max_depth', 3, 10),
'n_estimators': hp.randint('n_estimators', 100, 1000),
'num_parallel_tree': hp.randint('num_parallel_tree', 2, 8),
'min_child_weight': hp.randint('min_child_weight', 1, 10),
'subsample': hp.uniform('subsample', 0.5, 1),
'colsample_bytree': hp.uniform('colsample_bytree', 0.5, 1),
'colsample_bylevel': hp.uniform('colsample_bylevel', 0.5, 1),
'reg_lambda': hp.uniform('reg_lambda', 1e-3, 10),
'reg_alpha': hp.uniform('reg_alpha', 1e-3, 10),
'learning_rate': hp.uniform('learning_rate', 0.01, 0.3),
'gamma': hp.uniform('gamma', 0, 5),
}

ParameterSetups = []
TrainMSEValues = []
TestMSEValues = []

def objective(params):
    xgb_model = xgb.XGBRegressor(**params)
    cv = KFold(n_splits=20,shuffle=True)
    kfold_neg_mse = cross_val_score(estimator=xgb_model,X=X_trainer,y=y_trainer,scoring='neg_mean_squared_error',cv=cv,n_jobs=-1)
    kfold_mse = kfold_neg_mse * -1
    loss = np.average(kfold_mse) # MSE
    loss_variance = np.var(kfold_mse, ddof=1) # MSE variance

    ParameterSetups.append(params)
    TrainMSEValues.append(loss)
    xgb_model = xgb.XGBRegressor(**params)
    xgb_model = xgb_model.fit(X_trainer,y_trainer)
    y_pred = xgb_model.predict(X_tester)
    TestMSEValues.append(mean_squared_error(y_test,y_pred))

    return {'loss': loss, 'loss_variance': loss_variance, 'status': STATUS_OK}

trials = Trials()

best_params = fmin(objective, space, algo=tpe.suggest, max_evals=400, trials=trials,early_stop_fn=no_progress_loss(50))
best_params["tree_method"] = tree_methods[best_params["tree_method"]]
print("Best set of hyperparameters: ", best_params)

# Plotting Test Train Loss
epochs = []
SeriesOfBestTrainMSEValues = []
for count,i in enumerate(TrainMSEValues):
    epochs.append(count)
    if count == 0:
        SeriesOfBestTrainMSEValues.append(i)
        continue
    elif i <= SeriesOfBestTrainMSEValues[-1]:
        SeriesOfBestTrainMSEValues.append(i)
    else:
        SeriesOfBestTrainMSEValues.append(SeriesOfBestTrainMSEValues[-1])

SeriesOfBestTestMSEValues = []
for count,i in enumerate(TestMSEValues):
    if count == 0:
        SeriesOfBestTestMSEValues.append(i)
        continue
    elif i <= SeriesOfBestTestMSEValues[-1]:
        SeriesOfBestTestMSEValues.append(i)
    else:
        SeriesOfBestTestMSEValues.append(SeriesOfBestTestMSEValues[-1])
plt.plot(epochs,SeriesOfBestTrainMSEValues,c="red")
plt.plot(epochs,SeriesOfBestTestMSEValues,c="green")

 18%|█▊        | 73/400 [00:46<03:26,  1.58trial/s, best loss: 1.6100755039514456]
Best set of hyperparameters:  {'colsample_bylevel': np.float64(0.5013516099282204), 'colsample_bytree': np.float64(0.6624381901206082), 'gamma': np.float64(2.37758847937595), 'learning_rate': np.float64(0.0780011513915768), 'max_depth': np.int64(3), 'min_child_weight': np.int64(1), 'n_estimators': np.int64(892), 'num_parallel_tree': np.int64(4), 'reg_alpha': np.float64(0.11312116305417882), 'reg_lambda': np.float64(0.13660473903526457), 'subsample': np.float64(0.6601842077796963), 'tree_method': 'hist'}


NameError: name 'plt' is not defined

In [ ]:
# {'colsample_bylevel': np.float64(0.500782782175653), 'colsample_bytree': np.float64(0.7452614507731874), 'learning_rate': np.float64(0.1887773593372225), 'max_depth': np.int64(3), 'min_child_weight': np.int64(7), 'n_estimators': np.int64(572), 'num_parallel_tree': np.int64(4), 'reg_lambda': np.float64(8.689677547125394), 'subsample': np.float64(0.9251986368245226), 'tree_method': 'hist'}
# folds =       6
# max_evals =   100
# RMSE =	    1.499
# MAE =	        1.096
# MSE =         2.247

# {'colsample_bylevel': 0.6966086808300871, 'colsample_bytree': 0.8138666339638807, 'gamma': 0.7384107647391333, 'learning_rate': 0.22260089223414964, 'max_depth': 9, 'min_child_weight': 5, 'n_estimators': 748, 'num_parallel_tree': 2, 'reg_alpha': 0.08440472124495546, 'reg_lambda': 0.15356199239620577, 'subsample': 0.644588264012954, 'tree_method': 'hist'}
# folds =       10
# max_evals =   400
# no_progress = 50
# RMSE =	    1.486
# MAE =	        1.108
# MSE =         2.209

In [20]:
# xgb_model = xgb.XGBRegressor(**ParameterSetups[np.argmin(TestMSEValues)])
xgb_model = xgb.XGBRegressor(**{'colsample_bylevel': 0.6966086808300871, 'colsample_bytree': 0.8138666339638807, 'gamma': 0.7384107647391333, 'learning_rate': 0.22260089223414964, 'max_depth': 9, 'min_child_weight': 5, 'n_estimators': 748, 'num_parallel_tree': 2, 'reg_alpha': 0.08440472124495546, 'reg_lambda': 0.15356199239620577, 'subsample': 0.644588264012954, 'tree_method': 'hist'})
xgb_model.fit(X_train, y_train)
y_pred = xgb_model.predict(X_test)

rmse = root_mean_squared_error(y_test,y_pred)
mae = mean_absolute_error(y_test,y_pred)
mse = mean_squared_error(y_test,y_pred)
print(f"RMSE =\t{round(rmse,3)}")
print(f"MAE =\t{round(mae,3)}")
print(f"MSE =\t{round(mse,3)}")

RMSE =	1.508
MAE =	1.114
MSE =	2.274


In [21]:
plt.scatter(y_test,y_pred)
plt.plot([0,17],[0,17])
plt.ylim(0,17)
plt.xlim(0,17)
plt.xlabel(r"Observations, $y_{test}$")
plt.ylabel(r"Predictions, $y_{pred}$")

NameError: name 'plt' is not defined

In [22]:
y_residuals = y_test-y_pred
plt.scatter(y_test,y_residuals)
# plt.plot([0,17],[0,17])
plt.xlabel(r"Observations, $y_{test}$")
plt.ylabel(r"Residuals, $y_{test}-y_{pred}$")

NameError: name 'plt' is not defined

In [23]:
n = 25
iCoords_arr = np.linspace(0,2,n-1)
jCoords_arr = np.linspace(0,2,n-1)

ijCoords_lis = []
for i in iCoords_arr:
    for j in jCoords_arr:
        ijCoords_lis.append([i,j])
ijCoords_arr = np.array(ijCoords_lis)
y_pred_arr = xgb_model.predict(ijCoords_arr)

n_ci_bits = np.array(ijCoords_lis).T[0]
n_it_bits = np.array(ijCoords_lis).T[1]
plt.scatter(x=n_ci_bits,y=n_it_bits,c=y_pred_arr,cmap='viridis')

NameError: name 'plt' is not defined

In [26]:
np.average(y_pred_arr)

np.float32(11.953998)

# Save Model

In [ ]:
# xgb_model.save_model('ModelXG.json')